In [1]:
"""
This script test the dimensions of the VWR 96-well, 100ul, V-bottom plate
This plate in in the Hamilton MFX carrier 5 with 96-well holders (part number 182070).
"""

'\nThis script test the dimensions of the VWR 96-well, 100ul, V-bottom plate\nThis plate in in the Hamilton MFX carrier 5 with 96-well holders (part number 182070).\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
# from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL_filter,     # 1000 µL filtered 
    hamilton_96_tiprack_10uL_filter #Tip Rack with 96 10ul Low Volume Tip with filter
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

2026-02-17 10:04:24,534 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-17 10:04:24,541 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-17 10:04:24,544 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-17 10:04:27,708 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)
def VWR_96_wellplate_100_Vb_on_starCarrier_182070(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
  """
  
  return Plate(
    name=name,
    size_x=127.55,
    size_y=85.0,
    size_z=18.6,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb_on_starCarrier_182070.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11.5,  # measured
      dy=7.9,  # measured
      dz=2.15, # measured
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=16.7, # measured well depth
      material_z_thickness=1.0,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )


from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

In [6]:
###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000 = hamilton_96_tiprack_1000uL_filter("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = hamilton_96_tiprack_50uL_filter("tips_01") #  50 µL filter tips (slot-1)
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02") #10 ul filter tips
# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10


# STANDARDS RACK
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_19, rails=19)

# # labwareopentrons_24_tuberack_generic_1point5ml_snapcap_short
#dw_dilutions = BioER_96_wellplate_Vb_2200ul("dw_dilutions")
# dest_offset_x = (127.76 - tuberack_dest._size_x) / 2
# dest_offset_y = (85.48  - tuberack_dest._size_y) / 2

# adapter_dest = TubeRackAdapter(
#     name="dest_rack_adapter",
#     size_x=127.76,
#     size_y=85.48,
#     size_z=tuberack_dest._size_z,              # external height of the frame
#     model="tube_rack_adapter",
#     dx=dest_offset_x,
#     dy=dest_offset_y,
#     dz=0,
#     adapter_hole_size_x=tuberack_dest._size_x,
#     adapter_hole_size_y=tuberack_dest._size_y,
#     adapter_hole_size_z=tuberack_dest._size_z
# )
dwp_mod_dest.assign_child_resource(dw_dilutions)

# ----------carrier @ rail 13: water trough-------------------
# 96W, 100ul VWR PCR plate
dwp_mod_PCR = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_PCR")
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        0: dwp_mod_PCR,
        1: dwp_mod_trough,
    }
)
lh.deck.assign_child_resource(car_13, rails=13)


trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
dwp_mod_trough.assign_child_resource(trough)

# --- carrier @ rail-7: OT-2 tube rack with dsDNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

tuberack_src = opentrons_24_tuberack_generic_1point5ml_snapcap_short("src_rack")
src_offset_x = (127.76 - tuberack_src._size_x) / 2
src_offset_y = (85.48  - tuberack_src._size_y) / 2

adapter_src = TubeRackAdapter(
    name="src_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_src._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=src_offset_x,
    dy=src_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_src._size_x,
    adapter_hole_size_y=tuberack_src._size_y,
    adapter_hole_size_z=tuberack_src._size_z
)
adapter_src.assign_child_resource(tuberack_src)
dwp_mod_src.assign_child_resource(adapter_src)

CHANNEL_MM   = 6            # single channel we’ll use for the whole run
TIPRACK_50   = tiprack_50   # 50 µL filter tips (slot-1 on tip_car)


# await lh.pick_up_tips(TIPRACK_50["A1"], use_channels=[CHANNEL_MM])


ValueError: Resource with name 'tip_car' already defined.

In [ ]:
qPCR_plate = VWR_96_wellplate_100_Vb("qPCR_plate8")
dwp_mod_PCR.assign_child_resource(qPCR_plate)
st = 1
await lh.pick_up_tips(TIPRACK_50["A1"], use_channels=[CHANNEL_MM])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(qPCR_plate["A1"], vols=[0], use_channels=[CHANNEL_MM], settling_time=[st])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(qPCR_plate["H1"], vols=[0], use_channels=[CHANNEL_MM], settling_time=[st])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(qPCR_plate["A12"], vols=[0], use_channels=[CHANNEL_MM], settling_time=[st])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(qPCR_plate["H12"], vols=[0], use_channels=[CHANNEL_MM], settling_time=[st])
# await lh.prepare_for_manual_channel_operation(3)


In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[100], liquid_height=[4], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.dispense(DEST_PLATE["A7:H7"], vols=[50], liquid_height=[7], use_channels=CHANNELS_8, blow_out=[1])
# await lh.drop_tips(TIPRACK_50["A1"], use_channels=[6])
# await lh.discard_tips()
await lh.stop()
# await backend.stop() 